### Available Tables 
- s3_analytics_snapshots.dw_user_follow_history_quarterly
- dw_host_follow_history_quarterly also there, this has is_show_host=true.

In [0]:
%sql
select *
from s3_analytics_snapshots.dw_user_follow_history_quarterly
where snapshot_date = '2026-01-01'
limit 10;

In [0]:
%sql 
select * from s3_analytics_snapshots.dw_host_follow_history_quarterly 
--where is_show_host=true
limit 5

In [0]:
%sql 
select * from  raw_events r 
where event_date = '2026-03-01' and verb = 'follow'
limit 10

In [0]:
%sql 
select * from  raw_events r 
where event_date = '2026-03-01' and verb = 'follow'
limit 10

In [0]:

%sql 
WITH likes AS (
    SELECT actor_id, listing_id, event_date AS like_date
    FROM listing_actor_events_daily
    WHERE manual_likes > 0
    and event_date = '2026-04-01'
),
views AS (
    SELECT DISTINCT actor_id, listing_id, event_date AS view_date
    FROM listing_actor_events_daily
    WHERE listing_details_page_views > 0
    and event_Date between  '2026-02-01' and '2026-04-01'
)
SELECT
    ROUND(100.0 * COUNT(DISTINCT CASE WHEN v.actor_id IS NOT NULL 
                                      THEN l.actor_id||l.listing_id END)
        / NULLIF(COUNT(DISTINCT l.actor_id||l.listing_id), 0), 2) AS pct_likes_also_viewed
FROM likes l
LEFT JOIN views v
    ON l.actor_id = v.actor_id
    AND l.listing_id = v.listing_id
    AND v.view_date BETWEEN DATEADD(day, -60, l.like_date) AND l.like_date
